### UMAP

In this part, we will apply UMAP (Uniform Manifold Approximation and Projection) to the embeddings obtained from the CNN. These embeddings are vectors derived after flattening the outputs of the convolutional layers, effectively capturing the critical features of the images.

UMAP minimizes Cross-Entropy, which differs from t-SNE's KL-divergence, resulting in improved global structure preservation while still providing meaningful local relationships.

UMAP Documentation
Reference Paper: UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction (McInnes, L, Healy, J)


#### Importing Packages

In [ ]:
# Essential Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Machine Learning Libraries
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.decomposition import PCA

# Deep Learning Libraries
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Dimensionality Reduction
import umap

# Visualization Settings
sns.set(style='white', context='notebook', rc={'figure.figsize': (16, 10)})

#### Loading **MNIST** Dataset

In [ ]:
# Load MNIST data
mnist = fetch_openml('mnist_784')
X = mnist.data / 255.0  # Normalize pixel values to [0, 1]
y = LabelEncoder().fit_transform(mnist.target)  # Encode labels as integers
y = to_categorical(y, num_classes=10)  # One-hot encode labels
X = X.values.reshape(-1, 28, 28, 1)  # Reshape to (n_samples, height, width, channels)

#### Extracting **Embeddings** Using CNN

In this part, we aimed to extract the embeddings of the MNIST images — the vector representations obtained after flattening the outputs of the convolutional layers. For this, we implemented a Convolutional Neural Network (CNN) similar to the one used in PW11. The model was trained on a subset of the MNIST dataset, and its intermediate output (from the "flat" layer) was used to generate the embeddings.

**The steps involved:**

* Splitting the dataset into training and testing sets.
* Defining and training the CNN, which includes multiple convolutional layers, max-pooling layers, and a fully connected dense layer for classification.
* Extracting embeddings by creating a feature extraction model that outputs the "flat" layer’s activations.
* Storing these embeddings in a DataFrame along with their corresponding digit labels for downstream analysis.

These embeddings serve as compact, meaningful representations of the original images, capturing the essential features learned by the CNN.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the CNN model
input_layer = Input(shape=(28, 28, 1), name='l0')
l1 = Conv2D(9, (5, 5), padding='same', activation='relu', name='l1')(input_layer)
l1_mp = MaxPooling2D(pool_size=(2, 2), name='l1_mp')(l1)
l2 = Conv2D(9, (5, 5), padding='same', activation='relu', name='l2')(l1_mp)
l2_mp = MaxPooling2D(pool_size=(2, 2), name='l2_mp')(l2)
l3 = Conv2D(16, (3, 3), padding='same', activation='relu', name='l3')(l2_mp)
l3_mp = MaxPooling2D(pool_size=(2, 2), name='l3_mp')(l3)
flat = Flatten(name='flat')(l3_mp)
l4 = Dense(25, activation='relu', name='l4')(flat)
l5 = Dense(10, activation='softmax', name='l5')(l4)

model = Model(inputs=input_layer, outputs=l5)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the CNN
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))


In [ ]:
# Extract features (embeddings) from the 'flat' layer
feature_extractor = Model(inputs=model.input, outputs=model.get_layer('flat').output)
embeddings = feature_extractor.predict(X)  # Generate embeddings for all data

# Convert embeddings to DataFrame
embeddings_df = pd.DataFrame(embeddings)
embeddings_df['label'] = np.argmax(y, axis=1)  # Add labels for each embedding

#### Compute the 2D embedding with **UMAP**

In [ ]:
%%time
# Apply UMAP to the embeddings
print("Computing 2D UMAP on CNN embeddings...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings_df.iloc[:, :-1].values)

# Add UMAP results to the DataFrame
embeddings_df['umap-one'] = embedding_2d[:, 0]
embeddings_df['umap-two'] = embedding_2d[:, 1]

# Verify the updated DataFrame
print(embeddings_df.head())


In [ ]:
# Visualize UMAP projection
plt.figure(figsize=(16, 10))
sns.scatterplot(
    x='umap-one', 
    y='umap-two',
    hue='label',  # Ensure 'label' exists and matches the hue requirement
    data=embeddings_df,  # Correctly pass the DataFrame
    palette=sns.color_palette("hls", 10),
    legend="full",
    alpha=0.6
)
plt.title('UMAP Projection of CNN Embeddings')
plt.xlabel('UMAP-One')
plt.ylabel('UMAP-Two')
plt.show()


#### Compute the 3D embedding with **UMAP**

In [ ]:
# Apply UMAP for 3D projection
reducer_3d = umap.UMAP(random_state=42, n_components=3, n_neighbors=15, min_dist=0.1)
embedding_3d = reducer_3d.fit_transform(embeddings_df.iloc[:, :-1].values)  # Exclude label column

# Add UMAP results to DataFrame
embeddings_df['umap-3d-one'] = embedding_3d[:, 0]
embeddings_df['umap-3d-two'] = embedding_3d[:, 1]
embeddings_df['umap-3d-three'] = embedding_3d[:, 2]


In [ ]:
# 3D Visualization
fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    embeddings_df['umap-3d-one'],
    embeddings_df['umap-3d-two'],
    embeddings_df['umap-3d-three'],
    c=embeddings_df['label'],  # Ensure 'label' column exists
    cmap='tab10',
    alpha=0.7
)

ax.set_title('3D UMAP Projection of CNN Embeddings')
ax.set_xlabel('UMAP-3D-One')
ax.set_ylabel('UMAP-3D-Two')
ax.set_zlabel('UMAP-3D-Three')

plt.colorbar(scatter, ax=ax, shrink=0.5, aspect=10)
plt.show()


### Clustering in the embedded space

In [ ]:
# Function to cluster data and evaluate clustering metrics
def cluster_mnist(data, true_labels):
    kmeans = KMeans(n_clusters=np.unique(true_labels).size, random_state=42).fit(data)
    labels_preds = kmeans.predict(data)
    homogeneity = metrics.homogeneity_score(true_labels, labels_preds)
    completeness = metrics.completeness_score(true_labels, labels_preds)
    v_score = metrics.v_measure_score(true_labels, labels_preds)
    print(f"homogeneity | completeness | v_score")
    print(f"{homogeneity:>11.3f} | {completeness:>11.3f} | {v_score:>7.3f}")

In [ ]:
# Clustering in the embedding space
print("Clustering on CNN embeddings (flatten layer features):")
cluster_mnist(embeddings_df.iloc[:, :-1].values, embeddings_df['label'])  # Clustering on CNN embeddings

print("\nClustering on 2D UMAP embeddings:")
cluster_mnist(embedding_2d, embeddings_df['label'])  # Clustering on 2D UMAP embeddings

print("\nClustering on 3D UMAP embeddings:")
cluster_mnist(embedding_3d, embeddings_df['label'])  # Clustering on 3D UMAP embeddings


### Clustering results
As expected, clustering in a smaller embedded space gives much better results in cluster homogeneity and completness.
However, we can see already that more dimensions is not necesserly a guarantee in clustering quality when using a non-linear embedding process such as UMAP (KMeans use Euclidean distance metric)